# Chapitre 19 · La frontière (solutions des exercices)

Ce notebook contient **uniquement les réponses aux cinq exercices** du notebook
du chapitre. Le code de la leçon, lui, vit dans le notebook du chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut aussi pour les corrigés.

## Mise en place (reprise de la leçon)

Le strict nécessaire pour que les validations tournent de façon autonome : les imports,
puis le corpus jouet, RoPE et le petit GPT entraîné de la section 8 (pour l'exercice 5).

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
# rejoue les tirages aleatoires de la lecon (sections 2 et 4), pour retrouver
# EXACTEMENT le meme TinyGPT (et les memes perplexites) que dans le notebook du chapitre
_ = torch.randn(2), torch.randn(2)                                     # q, k de la demo RoPE
_ = torch.randn(1, 6, 512)                                             # x de la demo GQA
_ = nn.Linear(512, 512, bias=False), nn.Linear(512, 128, bias=False)   # W_q, W_k de la demo GQA

# petit corpus tres regulier : le contenu des sequences longues reste "connu",
# on isole ainsi l'effet de la POSITION.
motif = "le zemidjan de setondji roule vite. awa vend des mangues au marche. "
corpus = motif * 200

chars = sorted(set(corpus))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in corpus])
print(f"corpus : {len(corpus)} caracteres, vocabulaire de {V} tokens")

# --- RoPE : precalcul des cos/sin et application (convention rotate_half) ---
def rope_freqs(dim, L, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    pos = torch.arange(L).float()
    fr = torch.outer(pos, inv)
    emb = torch.cat([fr, fr], dim=-1)
    return emb.cos(), emb.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

# --- un GPT minuscule, une seule couche, RoPE dans l'attention ---
TRAIN_LEN = 24
d_model, n_head = 64, 4
d_head = d_model // n_head

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, d_model)
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)
        self.ff = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                                nn.Linear(4 * d_model, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, V, bias=False)

    def forward(self, idx, cos, sin):
        B, T = idx.shape
        h = self.emb(idx)
        x = self.ln1(h)
        q = self.Wq(x).view(B, T, n_head, d_head).transpose(1, 2)
        k = self.Wk(x).view(B, T, n_head, d_head).transpose(1, 2)
        v = self.Wv(x).view(B, T, n_head, d_head).transpose(1, 2)
        c = cos[:T].unsqueeze(0).unsqueeze(0)
        s = sin[:T].unsqueeze(0).unsqueeze(0)
        q = apply_rope(q, c, s)
        k = apply_rope(k, c, s)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(d_head)
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
        att = att.masked_fill(mask, float('-inf')).softmax(-1)
        o = (att @ v).transpose(1, 2).contiguous().view(B, T, d_model)
        h = h + self.Wo(o)
        h = h + self.ff(self.ln2(h))
        return self.head(h)

model = TinyGPT()
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
cos_tr, sin_tr = rope_freqs(d_head, TRAIN_LEN)

for step in range(600):
    ix = torch.randint(0, len(data) - TRAIN_LEN - 1, (32,))
    xb = torch.stack([data[i:i + TRAIN_LEN] for i in ix])
    yb = torch.stack([data[i + 1:i + TRAIN_LEN + 1] for i in ix])
    logits = model(xb, cos_tr, sin_tr)
    loss = F.cross_entropy(logits.reshape(-1, V), yb.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()

print(f"mise en place OK : modele entraine (loss {loss.item():.3f}), pret pour les validations")

### Exercice 1 · RMSNorm à la main — niveau ●

LayerNorm centre puis réduit. RMSNorm saute le centrage : elle divise simplement
par la RMS du vecteur. Calcule-la sur ce nouveau vecteur, sans regarder la section 3.

In [ ]:
x_ex = torch.tensor([1.0, -2.0, 3.0, 0.5, -1.5, 2.5, 0.0, -3.0])

rms_ex = torch.sqrt((x_ex ** 2).mean() + 1e-6)
rms_norm_ex = x_ex / rms_ex

print(f"RMS = {rms_ex.item():.4f}")
print(f"RMSNorm = {[round(v, 3) for v in rms_norm_ex.tolist()]}")

In [ ]:
# Validation : RMSNorm.
assert abs(rms_ex.item() - 1.9922) < 1e-3, "la RMS devrait valoir environ 1.9922"
assert abs(rms_norm_ex[2].item() - 1.506) < 1e-2, "verifie rms_norm_ex = x_ex / rms_ex"
print("OK : RMSNorm calculee (et remarque : pas besoin de la moyenne)")

### Exercice 2 · La propriété de distance de RoPE — niveau ●

Vérifie la propriété clé de RoPE sur une nouvelle paire de vecteurs : le
**produit scalaire** entre une requête tournée à la position `m` et une clé
tournée à la position `n` ne doit dépendre que de la **distance** `m - n`.

In [ ]:
q_ex = torch.tensor([0.8, -0.6])
k_ex = torch.tensor([0.3, 0.9])
theta_ex = 0.5

def rot_ex(v, ang):
    c, s = math.cos(ang), math.sin(ang)
    return torch.tensor([v[0] * c - v[1] * s, v[0] * s + v[1] * c])

def score_ex(m, n):
    return torch.dot(rot_ex(q_ex, m * theta_ex), rot_ex(k_ex, n * theta_ex)).item()

s_proche = score_ex(3, 7)     # distance -4
s_loin   = score_ex(13, 17)   # meme distance -4, positions decalees de 10
s_autre  = score_ex(3, 10)    # distance differente : -7
print(f"score(3, 7)   = {s_proche:+.4f}")
print(f"score(13, 17) = {s_loin:+.4f}")
print(f"score(3, 10)  = {s_autre:+.4f}")

In [ ]:
# Validation : meme distance => meme score ; distance differente => score different
assert abs(s_proche - s_loin) < 1e-5, "les scores a distance egale devraient etre identiques"
assert abs(s_proche - s_autre) > 1e-3, "une distance differente devrait changer le score"
print("OK : le score d'attention ne depend que de la distance m - n")

### Exercice 3 · Répéter les têtes K/V (GQA) — niveau ●●

GQA stocke peu de têtes K/V et les **répète** au moment du calcul pour servir
les têtes Q. Ici, 12 têtes de requête et 3 têtes K/V : chaque tête K/V doit
servir un groupe de 4 têtes Q.

In [ ]:
n_q_ex, n_kv_ex = 12, 3
K_ex = torch.randn(1, n_kv_ex, 5, 16)   # (batch, tetes K/V, seq, d_head)

K_rep_ex = K_ex.repeat_interleave(n_q_ex // n_kv_ex, dim=1)

print(f"K_ex shape     = {tuple(K_ex.shape)}")
print(f"K_rep_ex shape = {tuple(K_rep_ex.shape)}")

In [ ]:
# Validation : les groupes partagent bien la meme tete K/V
assert K_rep_ex.shape == (1, n_q_ex, 5, 16), "K_rep_ex doit avoir autant de tetes que Q"
assert torch.equal(K_rep_ex[0, 0], K_rep_ex[0, 3]), "les 4 premieres tetes doivent partager la meme K"
assert torch.equal(K_rep_ex[0, 8], K_rep_ex[0, 11]), "les 4 dernieres tetes doivent partager la meme K"
assert not torch.equal(K_rep_ex[0, 3], K_rep_ex[0, 4]), "la tete 4 appartient au groupe suivant"
print(f"OK : 1 tete K/V partagee par {n_q_ex // n_kv_ex} tetes Q, cache divise par {n_q_ex // n_kv_ex}")

### Exercice 4 · Le routage MoE — niveau ●●

Le routeur d'une couche MoE sort un score par expert. Transforme en probabilités,
garde les `k` meilleurs, renormalise leurs poids. Nouveaux scores, même mécanique
que la section 6.

In [ ]:
N_ex, k_top = 4, 2
logits_ex = torch.tensor([0.5, 2.5, 1.5, -1.0])   # le routeur score les 4 experts

probs_ex = F.softmax(logits_ex, dim=-1)
top_w_ex, top_i_ex = probs_ex.topk(k_top)
top_w_norm_ex = top_w_ex / top_w_ex.sum()

print(f"probas softmax  = {[round(v, 3) for v in probs_ex.tolist()]}")
print(f"top-{k_top} experts   = {top_i_ex.tolist()}   poids = {[round(v, 3) for v in top_w_norm_ex.tolist()]}")

In [ ]:
# Validation : routage top-2
assert sorted(top_i_ex.tolist()) == [1, 2], "les deux meilleurs experts sont 1 et 2"
assert abs(top_w_norm_ex.sum().item() - 1.0) < 1e-5, "les poids renormalises doivent sommer a 1"
assert abs(top_w_norm_ex[0].item() - 0.731) < 1e-2, "le poids du meilleur expert vaut environ 0.731"
print(f"OK : {k_top}/{N_ex} experts calcules pour ce token, les autres ne travaillent jamais")

### Exercice 5 · La perplexité au-delà de la longueur vue — niveau ●●●

Le petit GPT entraîné à la section 8 est encore en mémoire. Réécris toi-même la
mesure de perplexité : passe chaque bloc dans le modèle, accumule la NLL totale
et le nombre de tokens, et retrouve l'explosion puis la récupération partielle.

In [ ]:
@torch.no_grad()
def perplexite_ex(T, scale=1.0):
    model.eval()
    base = 10000.0 * (scale ** (d_head / (d_head - 2)))   # base RoPE etiree si scale > 1
    cos, sin = rope_freqs(d_head, T, base=base)
    total_nll, ntok = 0.0, 0
    n_blocs = min((len(data) - 1) // T, 20)
    for i in range(n_blocs):
        xb = data[i * T:(i + 1) * T].unsqueeze(0)
        yb = data[i * T + 1:(i + 1) * T + 1].unsqueeze(0)
        lo = model(xb, cos, sin)
        total_nll += F.cross_entropy(lo.reshape(-1, V), yb.reshape(-1), reduction='sum').item()
        ntok += yb.numel()
    return math.exp(total_nll / ntok)

ppl_vue_ex    = perplexite_ex(TRAIN_LEN, scale=1.0)
ppl_extra_ex  = perplexite_ex(TRAIN_LEN * 6, scale=1.0)   # 6x plus long, RoPE naif
ppl_scaled_ex = perplexite_ex(TRAIN_LEN * 6, scale=6.0)   # base RoPE etiree

print(f"perplexite a T = {TRAIN_LEN} (longueur vue)              : {ppl_vue_ex:.2f}")
print(f"perplexite a T = {TRAIN_LEN * 6} SANS scaling (extrapolation) : {ppl_extra_ex:.2f}")
print(f"perplexite a T = {TRAIN_LEN * 6} AVEC base RoPE etiree        : {ppl_scaled_ex:.2f}")

In [ ]:
# Validation : le cas qui echoue, puis la recuperation partielle
assert ppl_vue_ex < 1.5, "a la longueur vue, le modele devrait predire presque parfaitement"
assert ppl_extra_ex > 2 * ppl_vue_ex, "au-dela de la longueur d'entrainement, la perplexite doit exploser"
assert ppl_scaled_ex < ppl_extra_ex, "la base RoPE etiree doit recuperer une partie de la degradation"
print("OK : extrapolation naive = perplexite qui explose ; RoPE scaling = recuperation partielle")